# 🎙️ Whisper ASR — Live GPU FastAPI Server

Run your full **Whisper Large-v3-Turbo + FastAPI backend** on a free Google Colab GPU (NVIDIA T4).

Click the **Play (▶)** button below to start the server. It will automatically generate a **single clickable HTTPS link** (`https://xxxx.trycloudflare.com`) that connects your live Web Studio and Swagger Docs directly to the GPU!

In [ ]:
# 1. Setup GPU Environment & Clone Repo
import os, subprocess, time, re

print("⏳ Step 1/3: Checking GPU and setting up repository...")
!rm -rf whisper-asr-api
!git clone https://github.com/Omkar-Mohije/whisper-asr-api.git
%cd whisper-asr-api

# 2. Install dependencies
print("\n⏳ Step 2/3: Installing faster-whisper, PyTorch & FFmpeg...")
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!pip install -q --prefer-binary -r requirements.txt

# Install Cloudflare tunnel client (no account or login required)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null

# 3. Start FastAPI Server & Cloudflare Tunnel
print("\n🚀 Step 3/3: Booting FastAPI backend with GPU acceleration...")
server_proc = subprocess.Popen(["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print("\n" + "=" * 65)
        print("🎉 YOUR LIVE FASTAPI APP & WEB STUDIO IS RUNNING ON GPU!")
        print(f"👉 Tap here to open: {url}")
        print(f"👉 Swagger API Docs: {url}/docs")
        print("=" * 65 + "\n")
        print("Keep this Colab tab running while presenting.")
        break
